[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/skinning/skinning.ipynb)

# Linear Blend and Motor Skinning

This example computes the deformation of a cylindrical skin across a twisted joint under point-map blending and under motor blending. Each vertex of the skin follows the bones on either side of the joint according to its skinning weights. Averaging the destination points that each bone would carry a vertex to makes a twisted limb collapse at the waist like a wrung-out towel, whereas blending the rigid motions themselves keeps every cross-section circular at its rest radius.

In 3D projective geometric algebra, a bone's rigid motion is modelled as an even versor called a motor, whose sandwich action on points forms the linear map `motor >> Point`. Linear blend skinning averages those linear maps before evaluating at each vertex, whereas motor skinning either normalises a linear blend of the motors or interpolates along their screw geodesic through `(tip / root).log()` and `.exp()`, so both `motor >> vertex` deformations remain rigid motions on every cross-section.

In homogeneous matrix notation, each bone carries a $4 \times 4$ block matrix $M_i = \begin{bmatrix} R_i & \mathbf t_i \\ \mathbf 0^\top & 1 \end{bmatrix}$ and linear blend skinning evaluates $\bigl((1 - w) M_0 + w M_1\bigr)\begin{bmatrix}\mathbf p \\ 1\end{bmatrix}$. In dual-quaternion notation, each rigid motion is packed into $\hat q_i = q_{r,i} + \epsilon\,\tfrac{1}{2}(0, \mathbf t_i) q_{r,i}$ with $\epsilon^2 = 0$, the weighted sum $\hat b = (1 - w)\hat q_0 + w\hat q_1$ is divided by its dual norm $\|b_r\| + \epsilon\,(b_r \cdot b_\epsilon)/\|b_r\|$, and transforming a point unpacks the primal and dual parts $(b_w, \mathbf b_v)$ and $(e_w, \mathbf e_v)$ as $\mathbf p' = \mathbf p + 2\mathbf b_v \times (\mathbf b_v \times \mathbf p + b_w \mathbf p) + 2(b_w \mathbf e_v - e_w \mathbf b_v + \mathbf b_v \times \mathbf e_v)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np
from numga import NumpyContext
from numga.algebras import PGA3D as ga

from examples.util.animation import save_figure
from examples.surfaces.skinning import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Motor = ga.gatype.rotor()                           # Rotor+

## Blending motors and blending point maps

The rest skin is a unit cylinder along the x axis, with each ring weighted toward the tip bone in proportion to its axial position while the root bone stays fixed and the tip bone twists about the shared axis. Normalising a linear blend of the two motors projects each chord of the motor manifold back onto a rigid motion, while `(tip / root).log()` and `.exp()` travel along the screw arc itself at uniform angular speed. Averaging the linear point maps `motor >> Point` instead sends each vertex to the weighted midpoint of its two bone images, pulling the middle rings inward toward the chord across the cross-section.

In [ ]:
# The cylinder's rings along the bones and vertices around each, and how far the tip bone is twisted.
n_rings, n_around = 12, 24
twist_angle = np.radians(150.0)

In [ ]:
def cylinder(n_rings: int, n_around: int) -> tuple[Point, np.ndarray]:
    """Rotor turns around the x axis sweep a meridian from 0 to 1 into a unit-radius cylinder (`[n_rings * n_around] Point-`) with axial skinning weights."""
    weights, angles = np.meshgrid(
        np.linspace(0.0, 1.0, n_rings),
        np.linspace(0.0, 2 * np.pi, n_around, endpoint=False),
        indexing="ij",
    )
    # Turn the meridian line at y = 1, z = 0 around the x axis by each azimuth angle.
    meridian = mv.point(np.stack([weights, np.ones_like(weights), np.zeros_like(weights)], axis=-1))  # [n_rings, n_around] Point-
    turns = (mv.yz * (-angles / 2)).exp()                                                             # [n_rings, n_around] Motor+
    vertices = (turns >> meridian).reshape(-1)                                                        # [n_rings * n_around] Point-
    return vertices, weights.reshape(-1)


vertices, weights = cylinder(n_rings, n_around)                                # [n_rings * n_around] Point-, [n_rings * n_around]
root = mv.rotor()                                                              # [] Motor+
tip = (mv.yz * (twist_angle / 2)).exp()                                        # [] Motor+

In [ ]:
def blend_skin(vertices: Point, weights: np.ndarray, root: Motor, tip: Motor) -> tuple[Point, Point, Point]:
    """Normalised motor blending, motor slerp, and point-map blending each deform `vertices` (`[...] Point-`) between `root` and `tip`."""
    # Blend the bone motors linearly and project back onto the motor manifold by normalising.
    blended_motors = (root + weights * (tip - root)).normalized()                  # [...] Motor+
    motor_skin = blended_motors >> vertices                                        # [...] Point-

    # Slerp follows the screw geodesic through the relative motor's logarithm and exponential;
    # the normalised linear blend is a chord of the same arc and differs only in axial twist spacing.
    slerp_motors = ((tip / root).log() * weights).exp() * root                     # [...] Motor+
    slerp_skin = slerp_motors >> vertices                                          # [...] Point-

    # Linear blend skinning blends the two bones' point maps before evaluating at each vertex.
    blended_maps = (root >> Point) + weights * ((tip >> Point) - (root >> Point))  # [...] Point- <- Point-
    matrix_skin = blended_maps(vertices)                                           # [...] Point-
    return motor_skin, slerp_skin, matrix_skin


motor_skin, slerp_skin, matrix_skin = blend_skin(vertices, weights, root, tip) # [n_rings * n_around] Point-, Point-, Point-
save_figure(render.draw_skinning(motor_skin, slerp_skin, matrix_skin, n_rings, n_around), "skinning");

From left to right, the three panels show normalised motor blending, motor slerp, and point-map blending. Both motor skins preserve the rest cylinder's circular cross-section everywhere, with the coloured azimuth stripes bunching slightly toward the ends under the chord projection on the left and winding uniformly under the geodesic slerp in the centre, whereas the point-map skin on the right pinches toward the bone axis at the midpoint where both bones carry equal weight.

In [ ]:
# Joining the bone axis with a unit vertex forms the plane through both, whose norm is the radial distance.
for label, deformed in (("normalised motor blend", motor_skin), ("motor slerp", slerp_skin), ("point-map blend", matrix_skin)):
    radii = (mv.yz & deformed.normalized()).norm()                             # [n_rings * n_around] Scalar+
    print(f"{label}, minimum radius: {radii.to_array().min():.4f}")

In [ ]:
# checks
# Both motor blends keep every vertex at unit radius, whereas the point-map blend pulls a vertex of
# weight w inward onto the chord between its two bone images.
assert (((mv.yz & motor_skin.normalized()).norm() - 1).abs() < 1e-12).all()
assert (((mv.yz & slerp_skin.normalized()).norm() - 1).abs() < 1e-12).all()
chord_radius = np.sqrt(1 - 2 * weights * (1 - weights) * (1 - np.cos(twist_angle)))
assert (((mv.yz & matrix_skin.normalized()).norm() - chord_radius).abs() < 1e-12).all()